# ML1 - GridSearch with DecisionTreeRegressor, RandomForestRegressor

This notebook follows the ML1_Pipeline_Model notebook and focuses on:
1. Using DecisionTreeRegressor and RandomForestRegressor (or others)  
2. Hyperparameter tuning with GridSearchCV  
3. Cross-validation

## 1. Importing Libraries

In [ ]:
%conda install pandas numpy scikit-learn seaborn matplotlib


import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Lasso 
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import cross_val_score, train_test_split, GridSearchCV

## 2. Loading and Preparing Data with the Pipeline

We follow the same preparation steps as in the previous notebook.


In [ ]:
# Load the data from the pickle file.
df = pd.read_pickle('arbres_clean_Alexia.pkl')
df.dropna(subset=['anneedeplantation'], inplace=True)

#Prepare the train and test sets
X = df.drop(columns=['anneedeplantation'])
y = df['anneedeplantation']

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=20)

categorical_variables = X_train.select_dtypes(
    include=['object', 'string', 'category']
).columns.tolist()

numerical_variables = X_train.select_dtypes(include = ['number']).columns.tolist()
print(categorical_variables,numerical_variables)

#pipeline
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))])

numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())])

preprocessor = ColumnTransformer([
    ('cat', categorical_pipeline, categorical_variables),
    ('num', numerical_pipeline, numerical_variables)])
	
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

## 3. Initial Cross-Validation

Before tuning hyperparameters, let's evaluate a model with default parameters using cross-validation.


In [ ]:
# Initial cross-validation without hyperparameter tuning  
# We can also print the MSE or MAE scores from each cross-validation fold.

model_linear = Pipeline([
    ('preprocessor', preprocessor),
    ('regression', LinearRegression())
])
arbres_scores = cross_val_score(model_linear, X_train, y_train, cv = 5, scoring='neg_root_mean_squared_error') 

print("RMSE moyen :", np.mean(-arbres_scores))
print("RMSE écart-type moyen :", np.std(-arbres_scores))

## 4. 🌳 Modeling with GridSearchCV

Let's choose two machine learning models.

Evaluate the models with hyperparameter tuning (testing different hyperparameter values).  
Optimize the models as you wish, for example for Decision Tree and Random Forest:

### 🔍 Hyperparameters Tested
- `max_depth`: maximum depth of the trees [3, 4, 5, 10]  
- `min_samples_leaf`: minimum number of samples required at a leaf [0.1, 0.05]  
- `n_estimators` (for Random Forest only): number of trees in the forest [50, 100, 200]

Let's use GridSearchCV to automate the search for the best hyperparameters.

In [ ]:
# Hyperparameter tuning with GridSearchCV, and we can print the MSE or MAE scores for each cross-validation fold.
model_lasso = Pipeline([
    ('preprocessor', preprocessor),
    ('regression', Lasso(alpha=10, max_iter=10000))
])

param_grid = [
 {'regression__alpha': [0.001, 0.01, 0.1, 1, 3, 10, 100, 1000]}
 ]

#arbres_scores_lasso = cross_val_score(model_lasso, X_train, y_train, cv = 5, scoring='neg_mean_squared_error')
arbres_grid_search = GridSearchCV(model_lasso, param_grid, cv=10, scoring='neg_root_mean_squared_error')
arbres_grid_search.fit(X_train, y_train)

arbres_grid_results = pd.DataFrame(arbres_grid_search.cv_results_)

print(arbres_grid_results)
print(arbres_grid_search.best_estimator_)

#moyenne des rmse pour le meilleur alpha --> répond à la question de k-fold, pour le meilleur alpha
rmse_cv = -arbres_grid_search.best_score_
y_pred_test = arbres_grid_search.predict(X_test)
rmse_test = root_mean_squared_error(y_test, y_pred_test)

# Comparaison
print("Meilleur alpha :", arbres_grid_search.best_params_)
#estimation erreur de la validation croisée (k-fold)
print("RMSE validation croisée :", rmse_cv)
#estimation erreur réelle sur le jeu de test
print("RMSE test :", rmse_test)
#Pour info, différence entre les deux
print("Différence RMSE :", rmse_test - rmse_cv)

lasso_best = arbres_grid_search.best_estimator_

In [ ]:
# tuning random forest hyperparameters

# select a subset of the data for faster experimentation
X_train_sub = X_train.sample(frac=0.8, random_state=42)
y_train_sub = y_train.loc[X_train_sub.index]

# run one random forest with usual default hyperparameters for comparison
from sklearn.ensemble import RandomForestRegressor
model_rf_default = Pipeline([
    ('preprocessor', preprocessor),
    ('regression', RandomForestRegressor())
])
model_rf_default.fit(X_train_sub, y_train_sub)
y_pred_default = model_rf_default.predict(X_train_sub)


In [ ]:

from sklearn.metrics import root_mean_squared_error, mean_absolute_error
print("RMSE for default random forest on training subset:", root_mean_squared_error(y_train_sub, y_pred_default))
print("MAE for default random forest on training subset:", mean_absolute_error(y_train_sub, y_pred_default))


y_pred_test = model_rf_default.predict(X_test)
print("RMSE for default random forest on test subset:", root_mean_squared_error(y_test, y_pred_test))
print("MAE for default random forest on test subset:", mean_absolute_error(y_test, y_pred_test))

In [ ]:

# Hyperparameter tuning with GridSearchCV, and we can print the MSE or MAE scores for each cross-validation fold.
from sklearn.ensemble import RandomForestRegressor


model_random_forest = Pipeline([
    ('preprocessor', preprocessor),
    ('regression', RandomForestRegressor())
])

param_grid = [
 {'regression__n_estimators': [100, 200, 300],
  'regression__min_samples_leaf': [0.05, 0.1, 1, 10],
  'regression__max_depth': [None, 10, 20],
  'regression__random_state': [42]} 
 ]

#arbres_scores_lasso = cross_val_score(model_lasso, X_train, y_train, cv = 5, scoring='neg_mean_squared_error')
arbres_grid_search = GridSearchCV(model_random_forest, param_grid, cv=10, scoring='neg_root_mean_squared_error')
arbres_grid_search.fit(X_train_sub, y_train_sub)

arbres_grid_results = pd.DataFrame(arbres_grid_search.cv_results_)

print(arbres_grid_results)
print(arbres_grid_search.best_estimator_)


In [ ]:

#moyenne des rmse pour la meilleure combinaison de parametres --> répond à la question de k-fold, pour le meilleur alpha
rmse_cv = -arbres_grid_search.best_score_
y_pred_test = arbres_grid_search.predict(X_test)
rmse_test = root_mean_squared_error(y_test, y_pred_test)

print("RMSE validation croisée :", rmse_cv)
print("RMSE test :", rmse_test)
print("Différence RMSE :", rmse_test - rmse_cv)
print("Meilleure combinaison de parametres :", arbres_grid_search.best_params_)


In [ ]:
# tuning the randomforest on the whole test set with best hyperparameters
from sklearn.ensemble import RandomForestRegressor

model_rf = Pipeline([
    ('preprocessor', preprocessor),
    ('regression', RandomForestRegressor(**best_params_rf))
])
model_rf.fit(X_train, y_train)
y_pred_train = model_rf.predict(X_train)
print("RMSE for tuned random forest on training set:", root_mean_squared_error(y_train, y_pred_train))
print("MAE for tuned random forest on training set:", mean_absolute_error(y_train, y_pred_train))

# Comparaison
y_pred_test = model_rf.predict(X_test)
print("RMSE for tuned random forest on test set:", root_mean_squared_error(y_test, y_pred_test))
print("MAE for tuned random forest on test set:", mean_absolute_error(y_test, y_pred_test))



In [ ]:

print("Best model: ")
print(arbres_grid_search.best_params_)
random_forest_best = arbres_grid_search.best_params_

In [ ]:
# modeles avec best hyperparameter tuning

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_validate
from sklearn.tree import DecisionTreeRegressor
import numpy as np

model_lasso = Pipeline([
    ('preprocessor', preprocessor),
    ('regression', Lasso(**lasso_best.get_params()['regression']))
])

model_random_forest = Pipeline([
    ('preprocessor', preprocessor),
    ('regression', RandomForestRegressor(**random_forest_best.get_params()['regression']))
])

# model_decision_tree = Pipeline([
#     ('preprocessor', preprocessor),
#     ('regression', DecisionTreeRegressor())
# ])

scoring = {
    'rmse': 'neg_root_mean_squared_error',
    'mae': 'neg_mean_absolute_error',
    'r2': 'r2'
}


models = {
    'LinearRegression': model_linear,
    'Lasso': model_lasso,
    'RandomForest': model_random_forest,
    #'DecisionTree': model_decision_tree
}

for name, model in models.items():

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=5,
        scoring=scoring
    )

    rmse = -scores['test_rmse']
    mae = -scores['test_mae']
    r2 = scores['test_r2']

    # rmse = np.sqrt(mse)  # Not needed anymore since 'mse' is already RMSE

    print(f"\n{name}")
    print(f"RMSE moyen : {rmse.mean():.2f}")
    print(f"RMSE écart-type : {rmse.std():.2f}")
    print(f"MAE moyen  : {mae.mean():.2f}")
    print(f"R² moyen   : {r2.mean():.3f}")
    print(f"Écart-type MAE : {mae.std():.2f}")
    print(f"Écart-type R² : {r2.std():.3f}")
    print(f"Écart-type RMSE : {rmse.std():.2f}")
    print(f"RMSE min : {rmse.min():.2f}")
    print(f"RMSE max : {rmse.max():.2f}")
    print(f"MAE min : {mae.min():.2f}")
    print(f"MAE max : {mae.max():.2f}")
    print(f"R² min : {r2.min():.3f}")
    print(f"R² max : {r2.max():.3f}")

## 5. Final Evaluation on the Test Set

In [ ]:
# Code for evaluating the best model on the test set.

from sklearn.model_selection import cross_validate
import numpy as np

model_lasso = Pipeline([
    ('preprocessor', preprocessor),
    ('regression', Lasso(alpha=0.001, max_iter=10000))
])

scoring = {
    'rmse': 'neg_root_mean_squared_error',
    'mae': 'neg_mean_absolute_error',
    'r2': 'r2'
}

# Tes deux modèles Pipeline, déjà définis
models = {
    'LinearRegression': model_linear,
    'Lasso': model_lasso
}

for name, model in models.items():

    scores = cross_validate(
        model,
        X_test,
        y_test,
        cv=5,
        scoring=scoring
    )

    rmse = -scores['test_rmse']
    mae = -scores['test_mae']
    r2 = scores['test_r2']

    # rmse = np.sqrt(mse)  # Not needed anymore since 'mse' is already RMSE

    print(f"\n{name}")
    print(f"RMSE moyen : {rmse.mean():.2f}")
    print(f"RMSE écart-type : {rmse.std():.2f}")
    print(f"MAE moyen  : {mae.mean():.2f}")
    print(f"R² moyen   : {r2.mean():.3f}")
    print(f"Écart-type MAE : {mae.std():.2f}")
    print(f"Écart-type R² : {r2.std():.3f}")
    print(f"Écart-type RMSE : {rmse.std():.2f}")
    print(f"RMSE min : {rmse.min():.2f}")
    print(f"RMSE max : {rmse.max():.2f}")
    print(f"MAE min : {mae.min():.2f}")
    print(f"MAE max : {mae.max():.2f}")
    print(f"R² min : {r2.min():.3f}")
    print(f"R² max : {r2.max():.3f}")

## 6. Visualization of Predictions  
One model will suffice.

In [ ]:
model = model_lasso

model.fit(X_train, y_train)

y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# Visualisation of predictions.

plt.figure(figsize=(8, 6))

plt.scatter(y_test, y_pred_test, alpha=0.5)

# Línea de predicción perfecta
plt.plot([y_test.min(), y_test.max()],[y_test.min(), y_test.max()], 'r--')

plt.xlabel('Actual values')
plt.ylabel('Predicted values')
plt.title('Actual vs Predicted values')
plt.show()

# Visualisation of residuals.
residuals = y_test - y_pred_test
plt.figure(figsize=(8, 6))
plt.scatter(y_test, residuals, alpha=0.5)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Actual values')
plt.ylabel('Residuals')
plt.title('Residuals vs Actual values')
plt.show()



## [OPTIONAL] 7.Feature Importance

Using the random forest, how can we determine which features are the most important?

In [ ]:
# Code. In your opinion, how is feature importance calculated in Random Forest?

Typically, one might consider using backward elimination on feature importance to select the best subset of features, but this can be time-consuming if there are many features and the model takes a while to train, as is the case here. Instead, we can use Lasso-regularized regression to eliminate features, before retraining a RandomForest on the selected features. Create a pipeline with `LassoCV` as the selector and a RandomForest as the model with the best hyperparameters found by the previous grid-search. Evaluate on the test set.

In [ ]:
# Code

## 8. [OPTIONAL]  Compare more different models

Compare more regression models with one another (for example, in addition to linear regression, decision trees, and forests, use `GradientBoostingRegressor`, and the novel `TabICLRegressor` if you have time ( https://github.com/soda-inria/tabicl ))), by performing appropriate hyperparameter tuning for each model:

In [ ]:
# Code. Search which hyper-parameters are worth tuning for each model

Choose the model that performs best and its optimal hyperparameters, then select the best subset of features for this model, and finally perform one last grid search on your model’s hyperparameters using the selected features to obtain your final model. Evaluate it on the test set.

In [ ]:
# Code

## Conclusion